# Сбор 80 фотографий без бабочек из iNaturalist

В этом ноутбуке я собираю 80 фотографий, на которых нет взрослых бабочек. Они нужны, чтобы проверить ложные срабатывания моделей.

| Категория | Количество |
|---|---:|
| `other_insects` — другие насекомые | 40 |
| `nature_negative` — природные объекты | 40 |
| **Всего** | **80** |

Фотографии беру из iNaturalist с лицензией CC0. После отбора объединяю их с 320 фотографиями бабочек.

## 0. Подготовка

Устанавливаю библиотеки для получения, просмотра и проверки фотографий.

In [1]:
%pip install -q requests pandas pillow ipywidgets ImageHash


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Настройки

Задаю категории, источники фотографий и папки, в которых будут сохраняться результаты.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
from html import escape
from io import BytesIO
from itertools import combinations
import json
import time
import hashlib

import pandas as pd
import requests
import ipywidgets as widgets
from PIL import Image
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PROJECT_DIR = Path.home() / "butterfly-detection"
RUN_DIR = PROJECT_DIR / "negative_80"
DATA_DIR = RUN_DIR / "data"
IMAGES_DIR = RUN_DIR / "images"

POSITIVE_DIR = PROJECT_DIR / "positive_320_from_scratch"
POSITIVE_METADATA_PATH = POSITIVE_DIR / "data" / "metadata_positive.csv"
POSITIVE_IMAGES_DIR = POSITIVE_DIR / "images"

DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES_PATH = DATA_DIR / "candidates_negative.csv"
SELECTIONS_PATH = DATA_DIR / "selections_negative.csv"
FETCH_STATE_PATH = DATA_DIR / "negative_fetch_state.json"
TAXON_CACHE_PATH = DATA_DIR / "negative_taxon_ids.json"
METADATA_PATH = DATA_DIR / "metadata_negative.csv"
VALIDATION_PATH = DATA_DIR / "negative_validation.csv"
DUPLICATES_PATH = DATA_DIR / "duplicate_flags_negative.csv"
BBOX_NEGATIVE_PATH = DATA_DIR / "bbox_input_negative.csv"
BBOX_ALL_PATH = DATA_DIR / "bbox_input_all_400.csv"
METADATA_ALL_PATH = DATA_DIR / "metadata_all_400.csv"

API_URL = "https://api.inaturalist.org/v1/observations"
TAXA_URL = "https://api.inaturalist.org/v1/taxa/autocomplete"

PER_PAGE = 70

QUOTAS = {
    "other_insects": 40,
    "nature_negative": 40
}

TARGET_SELECTED = sum(QUOTAS.values())

SOURCES = [
    {
        "key": "dragonflies",
        "taxon": "Odonata",
        "group": "other_insects",
        "target": 120
    },
    {
        "key": "bees",
        "taxon": "Apidae",
        "group": "other_insects",
        "target": 120
    },
    {
        "key": "beetles",
        "taxon": "Coleoptera",
        "group": "other_insects",
        "target": 120
    },
    {
        "key": "flowers_asteraceae",
        "taxon": "Asteraceae",
        "group": "nature_negative",
        "target": 100
    },
    {
        "key": "flowers_rosaceae",
        "taxon": "Rosaceae",
        "group": "nature_negative",
        "target": 100
    },
    {
        "key": "foliage_fagaceae",
        "taxon": "Fagaceae",
        "group": "nature_negative",
        "target": 100
    },
    {
        "key": "foliage_ferns",
        "taxon": "Polypodiopsida",
        "group": "nature_negative",
        "target": 100
    },
    {
        "key": "other_plants",
        "taxon": "Poaceae",
        "group": "nature_negative",
        "target": 100
    }
]

CANDIDATE_COLUMNS = [
    "photo_id",
    "observation_id",
    "observation_url",
    "photo_url",
    "original_url",
    "license_code",
    "attribution",
    "source_key",
    "source_taxon"
]

SELECTION_COLUMNS = [
    "photo_id",
    "category",
    "reviewed_at"
]

retry = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

session = requests.Session()
session.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)

session.headers.update({
    "User-Agent": "CourseworkButterflyDataset/1.0 (academic research)"
})


def save_csv(table, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")

    table.to_csv(temp, index=False)
    temp.replace(path)


def load_candidates():
    if not CANDIDATES_PATH.exists():
        return pd.DataFrame(columns=CANDIDATE_COLUMNS)

    table = pd.read_csv(CANDIDATES_PATH)

    for col in CANDIDATE_COLUMNS:
        if col not in table.columns:
            table[col] = ""

    table["photo_id"] = pd.to_numeric(
        table["photo_id"],
        errors="coerce"
    )

    table = table.dropna(subset=["photo_id"]).copy()
    table["photo_id"] = table["photo_id"].astype("int64")

    return (
        table[CANDIDATE_COLUMNS]
        .drop_duplicates("photo_id")
        .reset_index(drop=True)
    )


def load_selections():
    if not SELECTIONS_PATH.exists():
        return pd.DataFrame(columns=SELECTION_COLUMNS)

    table = pd.read_csv(SELECTIONS_PATH).fillna("")

    for col in SELECTION_COLUMNS:
        if col not in table.columns:
            table[col] = ""

    table["photo_id"] = pd.to_numeric(
        table["photo_id"],
        errors="coerce"
    )

    table = table.dropna(subset=["photo_id"]).copy()
    table["photo_id"] = table["photo_id"].astype("int64")

    return (
        table[SELECTION_COLUMNS]
        .drop_duplicates("photo_id", keep="last")
    )


def load_positive():
    if not POSITIVE_METADATA_PATH.exists():
        return pd.DataFrame(
            columns=["photo_id", "observation_id"]
        )

    table = pd.read_csv(POSITIVE_METADATA_PATH)
    table["photo_id"] = table["photo_id"].astype("int64")

    return table


def original_url(url):
    url = str(url)

    for size in ("square", "thumb", "small", "medium", "large"):
        url = url.replace(
            f"/{size}.",
            "/original."
        )

    return url


def load_json(path, default):
    if path.exists():
        return json.loads(
            path.read_text(encoding="utf-8")
        )

    return default


def save_json(path, obj):
    temp = path.with_name(path.name + ".tmp")

    temp.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    temp.replace(path)


print("Папка отрицательных фотографий:", RUN_DIR)
print("Положительный датасет найден:", POSITIVE_METADATA_PATH.exists())
print("Цель:", TARGET_SELECTED, "фотографий без взрослых бабочек")

Папка отрицательных фотографий: /Users/ekaterinaseloceva/butterfly-detection/negative_80
Положительный датасет найден: True
Цель: 80 фотографий без взрослых бабочек


## 2. Проверка сохранённых данных

Смотрю, сколько кандидатов и решений уже есть, чтобы при повторном запуске продолжить работу с того же места.

In [3]:
print("Кандидатов:", len(load_candidates()))
print("Уже просмотрено:", len(load_selections()))
print("В положительном датасете:", len(load_positive()), "фотографий")

Кандидатов: 0
Уже просмотрено: 0
В положительном датасете: 320 фотографий


## 3. Получение фотографий-кандидатов

Получаю фотографии других насекомых и растений через API iNaturalist. Сохраняю ссылки и проверяю CC0 у каждой фотографии. Оригиналы пока не скачиваю.

In [4]:
def find_taxon_id(scientific_name):
    cache = load_json(TAXON_CACHE_PATH, {})

    if scientific_name in cache:
        return int(cache[scientific_name])

    response = session.get(
        TAXA_URL,
        params={"q": scientific_name},
        timeout=30
    )
    response.raise_for_status()

    matches = [
        item for item in response.json().get("results", [])
        if str(item.get("name", "")).lower()
        == scientific_name.lower()
    ]

    if not matches:
        raise ValueError(
            f"Не найден точный таксон {scientific_name}. "
            "Проверь его на iNaturalist."
        )

    taxon_id = int(matches[0]["id"])

    cache[scientific_name] = taxon_id
    save_json(TAXON_CACHE_PATH, cache)

    return taxon_id


def fetch_candidates(extra_per_source=0, max_pages_per_source=6):
    table = load_candidates()

    known_ids = set(table["photo_id"].tolist())

    positive = load_positive()

    positive_ids = (
        set(positive["photo_id"].tolist())
        if not positive.empty
        else set()
    )

    positive_observations = (
        set(positive["observation_id"].dropna().astype("int64"))
        if "observation_id" in positive
        else set()
    )

    state = load_json(FETCH_STATE_PATH, {})

    for source in SOURCES:
        key = source["key"]
        target = source["target"] + extra_per_source

        count = (
            int((table["source_key"] == key).sum())
            if not table.empty
            else 0
        )

        if count >= target:
            print(f"{key}: уже есть {count} кандидатов")
            continue

        try:
            taxon_id = find_taxon_id(source["taxon"])
        except Exception as exc:
            print(
                f"{key}: не удалось найти таксон — {exc}"
            )
            continue

        page = int(state.get(key, 1))

        for _ in range(max_pages_per_source):
            if count >= target:
                break

            params = {
                "taxon_id": taxon_id,
                "photos": "true",
                "photo_license": "cc0",
                "per_page": PER_PAGE,
                "page": page
            }

            try:
                response = session.get(
                    API_URL,
                    params=params,
                    timeout=30
                )
                response.raise_for_status()

                observations = (
                    response.json().get("results", [])
                )

            except Exception as exc:
                print(
                    f"{key}: не удалось загрузить "
                    f"страницу {page}: {exc}"
                )
                break

            if not observations:
                print(f"{key}: страницы закончились")
                break

            added = []

            for obs in observations:
                oid = obs.get("id")

                if (
                    oid is None
                    or int(oid) in positive_observations
                ):
                    continue

                for photo in obs.get("photos") or []:
                    if (
                        str(photo.get("license_code") or "").lower()
                        != "cc0"
                    ):
                        continue

                    pid = photo.get("id")
                    url = photo.get("url")

                    if (
                        pid is None
                        or not isinstance(url, str)
                        or not url.startswith("https://")
                    ):
                        continue

                    pid = int(pid)

                    if (
                        pid in known_ids
                        or pid in positive_ids
                    ):
                        continue

                    known_ids.add(pid)

                    added.append({
                        "photo_id": pid,
                        "observation_id": int(oid),
                        "observation_url": (
                            "https://www.inaturalist.org/"
                            f"observations/{oid}"
                        ),
                        "photo_url": url,
                        "original_url": original_url(url),
                        "license_code": "cc0",
                        "attribution": (
                            photo.get("attribution") or ""
                        ),
                        "source_key": key,
                        "source_taxon": source["taxon"]
                    })

            if added:
                table = pd.concat(
                    [table, pd.DataFrame(added)],
                    ignore_index=True
                )

                table = (
                    table.drop_duplicates("photo_id")
                    .reset_index(drop=True)
                )

                save_csv(table, CANDIDATES_PATH)

                count += len(added)

            print(
                f"{key}, стр. {page}: +{len(added)} фото, "
                f"всего по источнику {count}"
            )

            page += 1
            state[key] = page

            save_json(FETCH_STATE_PATH, state)

            time.sleep(1)

    print("Всего уникальных кандидатов:", len(table))

    return table


FETCH_NOW = True

if FETCH_NOW:
    df = fetch_candidates()
else:
    df = load_candidates()
    print(
        "Загрузка выключена. "
        "Сохранённых кандидатов:",
        len(df)
    )

dragonflies, стр. 1: +144 фото, всего по источнику 144
bees, стр. 1: +184 фото, всего по источнику 184
beetles, стр. 1: +154 фото, всего по источнику 154
flowers_asteraceae, стр. 1: +214 фото, всего по источнику 214
flowers_rosaceae, стр. 1: +131 фото, всего по источнику 131
foliage_fagaceae, стр. 1: +139 фото, всего по источнику 139
foliage_ferns, стр. 1: +163 фото, всего по источнику 163
other_plants, стр. 1: +186 фото, всего по источнику 186
Всего уникальных кандидатов: 1315


## 4. Проверка кандидатов

Проверяю, что все фотографии имеют разные ID и лицензию CC0. Отдельно исключаю совпадения с уже собранными фотографиями бабочек.

In [5]:
df = load_candidates()
positive = load_positive()

positive_ids = (
    set(positive["photo_id"].tolist())
    if not positive.empty
    else set()
)

print("Кандидатов:", len(df))
print("Уникальных ID:", df["photo_id"].nunique())

print(
    "Лицензии:",
    df["license_code"].value_counts(
        dropna=False
    ).to_dict() if not df.empty else {}
)

print(
    "Повторов с положительными фотографиями:",
    len(set(df["photo_id"]) & positive_ids)
)

if not df.empty:
    assert df["photo_id"].is_unique

    assert (
        df["license_code"].str.lower().eq("cc0").all()
    )

    assert (
        df["original_url"]
        .fillna("")
        .str.startswith("https://")
        .all()
    )

    assert not (
        set(df["photo_id"]) & positive_ids
    )

    display(df.head())

Кандидатов: 1315
Уникальных ID: 1315
Лицензии: {'cc0': 1315}
Повторов с положительными фотографиями: 0


,photo_id,observation_id,observation_url,photo_url,original_url,license_code,attribution,source_key,source_taxon
0,763102252,414515115,https://www.inaturalist.org/observations/41451...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved,dragonflies,Odonata
1,763099857,414513762,https://www.inaturalist.org/observations/41451...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved,dragonflies,Odonata
2,763074424,414501039,https://www.inaturalist.org/observations/41450...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved,dragonflies,Odonata
3,763074436,414501039,https://www.inaturalist.org/observations/41450...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved,dragonflies,Odonata
4,763072373,414500093,https://www.inaturalist.org/observations/41450...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved,dragonflies,Odonata


## 5. Ручной отбор фотографий

Просматриваю фотографии и выбираю `other_insects` или `nature_negative`. Если вижу взрослую бабочку, отклоняю снимок. Все решения сохраняются автоматически.

In [6]:
df = load_candidates()

selection_df = load_selections()

selections = {
    int(r["photo_id"]): r.to_dict()
    for _, r in selection_df.iterrows()
}

category_options = [
    ("Не выбрано", ""),
    ("Другие насекомые", "other_insects"),
    ("Природа без бабочек", "nature_negative"),
    ("Отклонить", "reject"),
    ("Нужно проверить", "unsure")
]

index_state = {"i": 0}

category_dropdown = widgets.Dropdown(
    options=category_options,
    description="Категория:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="440px")
)

info = widgets.HTML()
photo_html = widgets.HTML()
progress_html = widgets.HTML()
status = widgets.HTML()

prev_button = widgets.Button(
    description="← Назад"
)

save_button = widgets.Button(
    description="Сохранить и далее",
    button_style="success"
)

next_button = widgets.Button(
    description="Далее →"
)

unreviewed_button = widgets.Button(
    description="К нерассмотренной",
    button_style="info"
)


def category_counts():
    return {
        cat: sum(
            item.get("category") == cat
            for item in selections.values()
        )
        for cat in QUOTAS
    }


def preview_url(url):
    return (
        str(url)
        .replace("/square.", "/medium.")
        .replace("/original.", "/medium.")
    )


def show_photo():
    if df.empty:
        status.value = (
            "Нет кандидатов. Сначала получи их через API."
        )
        return

    row = df.iloc[index_state["i"]]
    pid = int(row["photo_id"])

    category_dropdown.value = (
        selections.get(pid, {}).get("category", "") or ""
    )

    counts = category_counts()

    info.value = (
        f"<b>Фото {index_state['i'] + 1} из {len(df)}</b> | "
        f"photo_id: <b>{pid}</b> | "
        f"отобрано: <b>{sum(counts.values())}/80</b>"
    )

    progress_html.value = " | ".join(
        f"{cat}: {counts[cat]}/{limit}"
        for cat, limit in QUOTAS.items()
    )

    obsid = int(row["observation_id"])

    duplicates = [
        int(r["photo_id"])
        for _, r in df.iterrows()
        if (
            int(r["observation_id"]) == obsid
            and int(r["photo_id"]) != pid
            and selections.get(
                int(r["photo_id"]), {}
            ).get("category") in QUOTAS
        )
    ]

    obs_warning = (
        f"<p style='color:#a16207'>"
        f"Из этого наблюдения уже выбраны фото: "
        f"{duplicates}</p>"
        if duplicates
        else ""
    )

    url = preview_url(row["photo_url"])
    original = str(row["original_url"])
    observation = str(row["observation_url"])

    photo_html.value = (
        f"<p>"
        f"<a href='{escape(original, quote=True)}' "
        f"target='_blank'>Открыть оригинал</a> | "
        f"<a href='{escape(observation, quote=True)}' "
        f"target='_blank'>Наблюдение iNaturalist</a>"
        f"</p>"
        f"<img src='{escape(url, quote=True)}' "
        f"style='max-width:700px; max-height:550px; "
        f"object-fit:contain;'>"
        + obs_warning
    )


def persist_selections():
    table = pd.DataFrame(
        list(selections.values()),
        columns=SELECTION_COLUMNS
    )

    save_csv(table, SELECTIONS_PATH)


def save_selection(_):
    if df.empty:
        return

    pid = int(df.iloc[index_state["i"]]["photo_id"])
    category = category_dropdown.value

    if not category:
        status.value = "<b>Выбери категорию.</b>"
        return

    old = selections.get(pid, {}).get("category")

    if (
        category in QUOTAS
        and category != old
        and category_counts()[category] >= QUOTAS[category]
    ):
        status.value = (
            f"<b>Квота {category} заполнена.</b>"
        )
        return

    selections[pid] = {
        "photo_id": pid,
        "category": category,
        "reviewed_at": datetime.now(
            timezone.utc
        ).isoformat()
    }

    persist_selections()

    status.value = (
        f"Сохранено: {pid} → {escape(category)}"
    )

    if index_state["i"] < len(df) - 1:
        index_state["i"] += 1

    show_photo()


def next_photo(_):
    index_state["i"] = min(
        len(df) - 1,
        index_state["i"] + 1
    )

    show_photo()


def prev_photo(_):
    index_state["i"] = max(
        0,
        index_state["i"] - 1
    )

    show_photo()


def go_unreviewed(_):
    for i, pid in enumerate(df["photo_id"]):
        if int(pid) not in selections:
            index_state["i"] = i
            show_photo()
            return

    status.value = (
        "Все кандидаты просмотрены. "
        "При необходимости получи новые через API."
    )


prev_button.on_click(prev_photo)
next_button.on_click(next_photo)
save_button.on_click(save_selection)
unreviewed_button.on_click(go_unreviewed)

display(
    widgets.VBox([
        info,
        photo_html,
        category_dropdown,
        widgets.HBox([
            prev_button,
            save_button,
            next_button,
            unreviewed_button
        ]),
        progress_html,
        status
    ])
)

for i, pid in enumerate(df["photo_id"]):
    if int(pid) not in selections:
        index_state["i"] = i
        break

show_photo()

## 6. Проверка прогресса

Смотрю, сколько фотографий выбрано в каждую категорию и сколько осталось до 80.

In [7]:
selection_df = load_selections()

counts = selection_df["category"].value_counts()

progress = pd.DataFrame({
    "category": list(QUOTAS),
    "target": list(QUOTAS.values())
})

progress["selected"] = (
    progress["category"]
    .map(counts)
    .fillna(0)
    .astype(int)
)

progress["remaining"] = (
    progress["target"] - progress["selected"]
)

display(progress)

print(
    "Всего отобрано:",
    progress["selected"].sum(),
    "/",
    TARGET_SELECTED
)

print(
    "Отклонено:",
    int(counts.get("reject", 0)),
    "| Нужно проверить:",
    int(counts.get("unsure", 0))
)

print(
    "Не просмотрено:",
    len(load_candidates()) - len(selection_df)
)

,category,target,selected,remaining
0,other_insects,40,40,0
1,nature_negative,40,40,0


Всего отобрано: 80 / 80
Отклонено: 0 | Нужно проверить: 0
Не просмотрено: 1235


## 7. Дополнительные кандидаты

Если подходящих фотографий мало, могу получить дополнительные. Уже выбранные фотографии и результаты отбора сохраняются.

In [8]:
ADD_MORE = False

if ADD_MORE:
    df = fetch_candidates(extra_per_source=60)
else:
    print(
        "Новые кандидаты не запрошены. "
        "При необходимости поставь ADD_MORE = True."
    )

Новые кандидаты не запрошены. При необходимости поставь ADD_MORE = True.


## 8. Список выбранных 80 фотографий

Когда выберу по 40 фотографий в каждую категорию, сохраняю итоговый список со ссылками на оригиналы.

In [9]:
selection_df = load_selections()

selected = selection_df[
    selection_df["category"].isin(QUOTAS)
].copy()

counts = selected["category"].value_counts().to_dict()

complete = all(
    counts.get(cat, 0) == limit
    for cat, limit in QUOTAS.items()
)

manifest = selected.merge(
    load_candidates(),
    on="photo_id",
    how="left",
    validate="one_to_one"
)

missing = int(manifest["original_url"].isna().sum())

print("Выбрано:", len(manifest), "/", TARGET_SELECTED)
print("По категориям:", counts)
print("Отсутствуют ссылки:", missing)

if complete and len(manifest) == TARGET_SELECTED and not missing:
    save_csv(manifest, METADATA_PATH)

    print(
        "Итоговый список сохранён:",
        METADATA_PATH
    )
else:
    print(
        "Пока не все квоты заполнены. "
        "Итоговый список не меняю."
    )

Выбрано: 80 / 80
По категориям: {'other_insects': 40, 'nature_negative': 40}
Отсутствуют ссылки: 0
Итоговый список сохранён: /Users/ekaterinaseloceva/butterfly-detection/negative_80/data/metadata_negative.csv


## 9. Скачивание и проверка оригиналов

После отбора скачиваю 80 оригиналов. Проверяю, что изображения открываются, и сохраняю результаты проверки.

In [10]:
def download_originals():
    if not METADATA_PATH.exists():
        print(
            "Нет metadata_negative.csv. "
            "Сначала заверши отбор."
        )
        return

    manifest = pd.read_csv(METADATA_PATH)

    selected_now = load_selections()

    selected_now = selected_now[
        selected_now["category"].isin(QUOTAS)
    ]

    if (
        len(manifest) != TARGET_SELECTED
        or set(manifest["photo_id"])
        != set(selected_now["photo_id"])
    ):
        print(
            "Итоговый список не совпадает с текущим "
            "отбором. Повтори шаг 8."
        )
        return

    history = {}

    if VALIDATION_PATH.exists():
        previous = pd.read_csv(VALIDATION_PATH)

        history = {
            int(row["photo_id"]): row.to_dict()
            for _, row in previous.iterrows()
        }

    results = history.copy()

    for i, row in manifest.iterrows():
        pid = int(row["photo_id"])
        url = str(row["original_url"])

        ext = Path(urlparse(url).path).suffix.lower()

        if ext not in (".jpg", ".jpeg", ".png", ".webp"):
            ext = ".jpg"

        path = IMAGES_DIR / f"{pid}{ext}"

        try:
            if not path.exists() or path.stat().st_size == 0:
                response = session.get(
                    url,
                    timeout=60,
                    stream=True
                )
                response.raise_for_status()

                temp = path.with_name(
                    path.name + ".part"
                )

                try:
                    with open(temp, "wb") as f:
                        for chunk in response.iter_content(
                            chunk_size=1024 * 1024
                        ):
                            if chunk:
                                f.write(chunk)

                    with Image.open(temp) as image:
                        image.verify()

                    temp.replace(path)

                finally:
                    response.close()

                    if temp.exists():
                        temp.unlink()

            with Image.open(path) as image:
                image.load()

                width = image.width
                height = image.height
                fmt = image.format

            digest = hashlib.sha256(
                path.read_bytes()
            ).hexdigest()

            results[pid] = {
                "photo_id": pid,
                "filename": path.name,
                "width": width,
                "height": height,
                "format": fmt,
                "size_mb": round(
                    path.stat().st_size / (1024 ** 2),
                    2
                ),
                "sha256": digest,
                "valid": True,
                "error": ""
            }

        except Exception as exc:
            results[pid] = {
                "photo_id": pid,
                "filename": path.name,
                "width": None,
                "height": None,
                "format": "",
                "size_mb": None,
                "sha256": "",
                "valid": False,
                "error": str(exc)
            }

        save_csv(
            pd.DataFrame(list(results.values())),
            VALIDATION_PATH
        )

        if (
            (i + 1) % 10 == 0
            or i + 1 == len(manifest)
        ):
            seen = set(
                manifest.iloc[:i + 1]["photo_id"].astype(int)
            )

            good = sum(
                str(results[pid]["valid"]).lower() == "true"
                for pid in seen
            )

            print(
                f"Проверено: {i + 1}/{len(manifest)}; "
                f"успешно: {good}"
            )

        time.sleep(0.1)

    print("Результаты:", VALIDATION_PATH)


DOWNLOAD_ORIGINALS = True

if DOWNLOAD_ORIGINALS:
    download_originals()
else:
    print(
        "Скачивание выключено. "
        "Когда отберёшь 80 фото, "
        "установи DOWNLOAD_ORIGINALS = True."
    )

Проверено: 10/80; успешно: 10
Проверено: 20/80; успешно: 20
Проверено: 30/80; успешно: 30
Проверено: 40/80; успешно: 40
Проверено: 50/80; успешно: 50
Проверено: 60/80; успешно: 60
Проверено: 70/80; успешно: 70
Проверено: 80/80; успешно: 80
Результаты: /Users/ekaterinaseloceva/butterfly-detection/negative_80/data/negative_validation.csv


## 10. Проверка похожих фотографий

Проверяю, нет ли одинаковых или очень похожих изображений среди выбранных 80 фотографий и уже собранных 320.

In [11]:
import imagehash

if not (
    METADATA_PATH.exists()
    and VALIDATION_PATH.exists()
):
    print(
        "Сначала сформируй список 80 фотографий "
        "и скачай оригиналы."
    )

else:
    manifest = pd.read_csv(METADATA_PATH)
    valid = pd.read_csv(VALIDATION_PATH)

    valid = valid[
        (
            valid["valid"]
            .astype(str)
            .str.lower() == "true"
        )
        & valid["photo_id"].isin(manifest["photo_id"])
    ].copy()

    negative_hashes = {}
    negative_sha = {}

    for _, item in valid.iterrows():
        pid = int(item["photo_id"])
        path = IMAGES_DIR / str(item["filename"])

        if not path.exists():
            continue

        with Image.open(path) as image:
            negative_hashes[pid] = imagehash.phash(
                image.convert("RGB")
            )

        negative_sha[pid] = str(item["sha256"])

    positive_hashes = {}
    positive_sha = {}

    positive = load_positive()

    if POSITIVE_IMAGES_DIR.exists() and not positive.empty:
        for pid in positive["photo_id"]:
            matches = list(
                POSITIVE_IMAGES_DIR.glob(f"{int(pid)}.*")
            )

            matches = [
                p for p in matches
                if p.suffix.lower()
                in (".jpg", ".jpeg", ".png", ".webp")
            ]

            if not matches:
                continue

            path = matches[0]

            try:
                with Image.open(path) as image:
                    positive_hashes[int(pid)] = imagehash.phash(
                        image.convert("RGB")
                    )

                positive_sha[int(pid)] = hashlib.sha256(
                    path.read_bytes()
                ).hexdigest()

            except Exception as exc:
                print(
                    "Не удалось проверить положительное фото",
                    pid,
                    exc
                )

    flags = []

    for a, b in combinations(negative_hashes, 2):
        dist = negative_hashes[a] - negative_hashes[b]
        exact = negative_sha[a] == negative_sha[b]

        if dist <= 6 or exact:
            flags.append({
                "negative_photo_id": a,
                "other_photo_id": b,
                "part": "negative",
                "phash_distance": dist,
                "identical_file": exact
            })

    for a in negative_hashes:
        for b in positive_hashes:
            dist = negative_hashes[a] - positive_hashes[b]
            exact = negative_sha[a] == positive_sha[b]

            if dist <= 6 or exact:
                flags.append({
                    "negative_photo_id": a,
                    "other_photo_id": b,
                    "part": "positive",
                    "phash_distance": dist,
                    "identical_file": exact
                })

    flags_df = pd.DataFrame(
        flags,
        columns=[
            "negative_photo_id",
            "other_photo_id",
            "part",
            "phash_distance",
            "identical_file"
        ]
    )

    save_csv(flags_df, DUPLICATES_PATH)

    print(
        "Проверено отрицательных оригиналов:",
        len(negative_hashes),
        "/ 80"
    )

    print(
        "Проверено положительных оригиналов:",
        len(positive_hashes),
        "/ 320"
    )

    print(
        "Найдено подозрительно похожих пар:",
        len(flags_df)
    )

    display(flags_df.head(30))

    repeated = manifest[
        manifest.duplicated(
            "observation_id",
            keep=False
        )
    ].sort_values("observation_id")

    print(
        "Фотографий из наблюдений "
        "с несколькими кадрами:",
        len(repeated)
    )

    display(
        repeated[
            ["photo_id", "observation_id", "category"]
        ].head(30)
    )

    if len(positive_hashes) < 320:
        print(
            "Не все 320 оригиналов доступны локально. "
            "Полная межгрупповая проверка не выполнена."
        )

    print(
        "Подозрительные пары нужно просмотреть вручную; "
        "pHash не гарантирует отсутствие похожих снимков."
    )

Проверено отрицательных оригиналов: 80 / 80
Проверено положительных оригиналов: 320 / 320
Найдено подозрительно похожих пар: 0


,negative_photo_id,other_photo_id,part,phash_distance,identical_file


Фотографий из наблюдений с несколькими кадрами: 14


,photo_id,observation_id,category
58,761946857,413944475,nature_negative
59,761946855,413944475,nature_negative
53,762264087,414126515,other_insects
54,762264084,414126515,nature_negative
35,762678484,414305217,nature_negative
36,762678504,414305217,nature_negative
25,762812806,414372518,other_insects
26,762812834,414372518,other_insects
67,763037738,414482821,nature_negative
68,763037732,414482821,nature_negative


Подозрительные пары нужно просмотреть вручную; pHash не гарантирует отсутствие похожих снимков.


## 11. Подготовка общего датасета из 400 фотографий

Проверяю итоговые 80 фотографий и объединяю их с уже собранными 320. Сохраняю метаданные и общий список ссылок для bbox.

In [12]:
if not (
    METADATA_PATH.exists()
    and VALIDATION_PATH.exists()
    and POSITIVE_METADATA_PATH.exists()
):
    print(
        "Не хватает итоговых данных. "
        "Проверь шаги 8–10 и наличие metadata_positive.csv."
    )

else:
    negative = pd.read_csv(METADATA_PATH)
    positive = load_positive()
    checks = pd.read_csv(VALIDATION_PATH)

    counts = negative["category"].value_counts().to_dict()

    exact_quota = (
        all(
            counts.get(cat, 0) == n
            for cat, n in QUOTAS.items()
        )
        and len(negative) == 80
    )

    all_valid = (
        checks.set_index("photo_id")
        .reindex(negative["photo_id"])["valid"]
        .astype(str)
        .str.lower()
        .eq("true")
        .all()
    )

    files_exist = all(
        any(IMAGES_DIR.glob(f"{int(pid)}.*"))
        for pid in negative["photo_id"]
    )

    unique = (
        negative["photo_id"].is_unique
        and positive["photo_id"].is_unique
    )

    no_overlap = set(
        negative["photo_id"]
    ).isdisjoint(
        set(positive["photo_id"])
    )

    all_cc0 = (
        negative["license_code"].str.lower().eq("cc0").all()
        and positive["license_code"].str.lower().eq("cc0").all()
    )

    correct_positive_count = len(positive) == 320

    exact_duplicates = False

    if DUPLICATES_PATH.exists():
        flags = pd.read_csv(DUPLICATES_PATH)

        exact_duplicates = (
            flags["identical_file"]
            .astype(str)
            .str.lower()
            .eq("true")
            .any()
            if not flags.empty
            else False
        )

    print("Квоты 40/40:", exact_quota)
    print("Все 80 оригиналов проверены:", all_valid)
    print("Все файлы на месте:", files_exist)
    print("320 положительных фото:", correct_positive_count)
    print(
        "Уникальные ID, нет пересечений:",
        unique and no_overlap
    )
    print("Лицензия CC0 у всех 400:", all_cc0)
    print(
        "Нет полностью одинаковых файлов:",
        not exact_duplicates
    )

    if all([
        exact_quota,
        all_valid,
        files_exist,
        correct_positive_count,
        unique,
        no_overlap,
        all_cc0,
        not exact_duplicates
    ]):
        negative_bbox = negative[
            ["photo_id", "original_url"]
        ].rename(
            columns={"original_url": "image_url"}
        )

        positive_bbox = positive[
            ["photo_id", "original_url"]
        ].rename(
            columns={"original_url": "image_url"}
        )

        full_bbox = pd.concat(
            [positive_bbox, negative_bbox],
            ignore_index=True
        )

        assert (
            len(full_bbox) == 400
            and full_bbox["photo_id"].is_unique
        )

        save_csv(
            negative_bbox,
            BBOX_NEGATIVE_PATH
        )

        save_csv(
            full_bbox,
            BBOX_ALL_PATH
        )

        negative_metadata = negative.assign(
            dataset_part="negative"
        )

        positive_metadata = positive.assign(
            dataset_part="positive"
        )

        full_metadata = pd.concat(
            [positive_metadata, negative_metadata],
            ignore_index=True
        )

        save_csv(
            full_metadata,
            METADATA_ALL_PATH
        )

        print(
            "Общий список для bbox:",
            BBOX_ALL_PATH
        )

        print(
            "Общие метаданные:",
            METADATA_ALL_PATH
        )

        print("Фотографий:", len(full_bbox))

    else:
        print(
            "Экспорт не выполнен: "
            "сначала исправь замечания выше."
        )

Квоты 40/40: True
Все 80 оригиналов проверены: True
Все файлы на месте: True
320 положительных фото: True
Уникальные ID, нет пересечений: True
Лицензия CC0 у всех 400: True
Нет полностью одинаковых файлов: True
Общий список для bbox: /Users/ekaterinaseloceva/butterfly-detection/negative_80/data/bbox_input_all_400.csv
Общие метаданные: /Users/ekaterinaseloceva/butterfly-detection/negative_80/data/metadata_all_400.csv
Фотографий: 400


## Результат

После отбора у меня будут 80 проверенных фотографий без взрослых бабочек. Вместе с положительными снимками это даст 400 изображений для bbox и review. Все ссылки, категории и исходные данные сохраняются отдельно.